# Notebook 1 — Data Collection & Preview
## NFL Player Performance Analysis

**Authors:** Milan Jovkić, Uroš Petrašković  
**Course:** Analiza i Obrada Podataka  
**Date:** 2025

---

This notebook documents **how the data was collected**, the **sources used**, and provides a **preview of all datasets** used in the project. We cover four NFL positions:

| Position | Abbreviation | Data Source | Method |
|----------|-------------|-------------|--------|
| Quarterback | QB | Pro Football Reference | Selenium web scraper |
| Running Back | RB | Pro Football Reference | Selenium web scraper |
| Tight End | TE | Pro Football Reference | Selenium web scraper |
| Wide Receiver | WR | HuggingFace / nflverse | Pre-built dataset |

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded successfully.")

---
## 1. Data Sources Overview

Our project uses data from **two primary sources**:

### 1.1 Pro Football Reference (QB, RB, TE)

[Pro Football Reference](https://www.pro-football-reference.com) (PFR) is the gold standard for NFL statistics. We scraped individual player pages to collect season-level stats across multiple statistical categories.

**Why PFR?**
- Most comprehensive NFL stats database
- Includes advanced analytics (air yards, broken tackles, pressure rates)
- Historical data going back decades
- Regularly updated during the season

### 1.2 HuggingFace / nflverse (WR)

For Wide Receivers, we used a pre-built dataset from HuggingFace:  
🔗 [SebastianAndreu/24679_NFL_WR_Dataset_2025](https://huggingface.co/datasets/SebastianAndreu/24679_NFL_WR_Dataset_2025)

This dataset was built from **nflverse** play-by-play data and includes:
- Game-level statistics (2015–2025)
- Advanced metrics (EPA, WPA, air yards, YAC)
- Situational splits (red zone, quarter, win probability)
- Weather data (temperature, wind, humidity)

We aggregated the weekly data into season-level summaries using `scripts/combine_wr_master.py`.

---
## 2. Web Scraping Pipeline (`scrapers/` folder)

The scraping pipeline uses **Selenium WebDriver** with **Opera GX** to bypass Cloudflare protection on Pro Football Reference.

### Architecture

```
scrapers/
├── nfl_qb_scraper.py    → 74 quarterbacks
├── nfl_rb_scraper.py    → 77 running backs
└── nfl_te_scraper.py    → 27 tight ends
```

### How It Works

1. **Player list** — Each scraper has a hardcoded list of `(player_id, player_name)` tuples  
2. **URL construction** — Player pages follow the pattern:  
   `https://www.pro-football-reference.com/players/{first_letter}/{player_id}.htm`  
3. **Cloudflare bypass** — Opera GX with anti-automation flags, 6–7 second delays between requests  
4. **Table extraction** — BeautifulSoup parses HTML tables using `data-stat` attributes (some tables hidden in HTML comments)
5. **Column mapping** — Raw `data-stat` identifiers are mapped to human-readable column names
6. **Per-player CSV output** — Each player's stats are saved to individual CSV files in `data/raw/{position}/`

### Tables Scraped Per Position

| Position | Tables | Key Stats |
|----------|--------|-----------|
| QB | 7 tables | Passing, Adjusted Passing, Advanced Passing, Rushing/Receiving, Adv Rush/Rec, Defense/Fumbles, Snap Counts |
| RB | 4 tables | Rushing/Receiving, Advanced Rush/Rec, Defense/Fumbles, Snap Counts |
| TE | 4 tables | Receiving/Rushing, Advanced Rec/Rush, Defense/Fumbles, Snap Counts |

### Rate Limiting & Error Handling
- **6–7 second random delay** between players
- **3 retry attempts** with 8–15 second backoff for Cloudflare challenges
- **Skip already-scraped** players (checks for existing CSVs)
- Graceful error handling per player (one failure doesn't stop the batch)

### 2.1 Scraper Code Highlights

Below we show the key structures from our QB scraper (the others follow the same pattern):

In [ ]:
# Show the list of QBs we scraped (from scrapers/nfl_qb_scraper.py)
qb_list_sample = [
    ('MahoPa00', 'Patrick Mahomes', 1995),
    ('BradTo00', 'Tom Brady', 1977),
    ('AlleJo02', 'Josh Allen', 1996),
    ('JackLa00', 'Lamar Jackson', 1997),
    ('BurrJo01', 'Joe Burrow', 1996),
    ('HerbJu00', 'Justin Herbert', 1998),
    ('RodgAa00', 'Aaron Rodgers', 1983),
    ('ManpPe00', 'Peyton Manning', 1976),
    ('BreeDr00', 'Drew Brees', 1979),
]

print(f"Example QBs scraped (showing 9 of 74):")
print(f"{'Player ID':<14} {'Name':<25} {'Birth Year'}")
print("-" * 50)
for pid, name, yr in qb_list_sample:
    print(f"{pid:<14} {name:<25} {yr}")

print(f"\nURL example: https://www.pro-football-reference.com/players/M/MahoPa00.htm")

In [ ]:
# The COLUMN_MAPPING translates raw HTML data-stat attributes to clean names
# Example mappings from the QB scraper:
column_mapping_sample = {
    'pass_cmp': 'Cmp',        'pass_att': 'Att',
    'pass_cmp_pct': 'Cmp%',   'pass_yds': 'Yds',
    'pass_td': 'TD',          'pass_int': 'Int',
    'pass_rating': 'Rate',    'qbr': 'QBR',
    'pass_sacked': 'Sk',      'pass_first_down': '1D',
    'pass_air_yds': 'AirYds', 'pass_yac': 'YAC',
    'rush_att': 'Rush_Att',   'rush_yds': 'Rush_Yds',
}

print("Column Mapping (HTML data-stat → Clean Name):")
print(f"{'Raw Attribute':<28} → {'Clean Name'}")
print("-" * 45)
for raw, clean in column_mapping_sample.items():
    print(f"{raw:<28} → {clean}")

---
## 3. Data Combining Pipeline (`scripts/` folder)

After scraping, individual player CSVs are merged into **master tables** — one per position.

```
scripts/
├── combine_qb_master.py    → data/fully combined/qb_master.csv
├── combine_rb_master.py    → data/fully combined/rb_master.csv
├── combine_te_master.py    → data/fully combined/te_master.csv
└── combine_wr_master.py    → data/fully combined/wr_all_seasons.csv + wr_all_weeks.csv
```

### Combining Strategy (QB example)

The QB master table merges **7 stat table types** using column prefixes to avoid name collisions:

| Source Table | Prefix | Example Column |
|--------------|--------|----------------|
| passing.csv | *(none)* | `Yds`, `TD`, `Cmp%` |
| adjusted_passing.csv | `adj_` | `adj_ANY/A+`, `adj_Rate+` |
| advanced_passing.csv | `adv_pass_` | `adv_pass_pass_air_yds`, `adv_pass_pocket_time` |
| rushing_receiving.csv | `rr_` | `rr_Rush_Yds`, `rr_Rec_Yds` |
| adv_rushing_receiving.csv | `adv_rr_` | `adv_rr_Rush_YBC`, `adv_rr_Rec_BrkTkl` |
| defense_fumbles.csv | `def_` | `def_FF`, `def_Fmb` |
| snap_counts.csv | `snp_` | `snp_offense`, `snp_Off%` |

**Merge keys:** `Season`, `Player`, `PlayerID`  
**Shared columns** (kept once): `Age`, `Team`, `Pos`, `G`, `GS`

### QB Elo/Rankings Data

The QB master also merges **quarterback Elo ratings** from a separate rankings dataset:
- Source: `data/nfl elo data/qb_rankings_career.csv`
- Columns prefixed with `elo_` (e.g., `elo_QB Elo`, `elo_CPOE`)
- Matched on `Player` + `Season`

### WR Combining (Different Approach)

Since WR data comes from nflverse (week-level), the combine script:
1. Loads all years (2015–2025) of weekly CSVs
2. Saves the raw weekly data as `wr_all_weeks.csv` (46,115 rows)
3. Aggregates to **per-player per-season** summaries → `wr_all_seasons.csv` (5,529 rows)
   - Counting stats (targets, yards, TDs) are **summed**
   - Rate stats (EPA, catch rate, target share) are **averaged**

---
## 4. Loading the Master Datasets

Let's load all four position datasets and examine their structure:

In [ ]:
# Load all master datasets
qb = pd.read_csv('data/fully combined/qb_master.csv')
rb = pd.read_csv('data/fully combined/rb_master.csv')
te = pd.read_csv('data/fully combined/te_master.csv')
wr = pd.read_csv('data/fully combined/wr_all_seasons.csv')
wr_weeks = pd.read_csv('data/fully combined/wr_all_weeks.csv')

# Also load ELO data
elo_career = pd.read_csv('data/nfl elo data/qb_rankings_career.csv')
elo_2025   = pd.read_csv('data/nfl elo data/qb_rankings_2025.csv')

datasets = {
    'QB Master': qb, 'RB Master': rb, 'TE Master': te,
    'WR Seasons': wr, 'WR Weeks': wr_weeks,
    'ELO Career': elo_career, 'ELO 2025': elo_2025
}

print(f"{'Dataset':<20} {'Rows':>8} {'Cols':>6} {'Size (KB)':>10}")
print("=" * 48)
for name, df in datasets.items():
    size_kb = df.memory_usage(deep=True).sum() / 1024
    print(f"{name:<20} {len(df):>8,} {len(df.columns):>6} {size_kb:>10,.1f}")

---
## 5. Quarterback (QB) Dataset — Preview

**Source:** Pro Football Reference (scraped)  
**Players:** 74 QBs | **Seasons:** 1979–2025 | **Columns:** 142

The QB dataset is the most comprehensive, spanning 46 seasons with 142 features covering:
- **Core passing** — completions, attempts, yards, TDs, interceptions, passer rating
- **Adjusted passing** — era-adjusted metrics (ANY/A+, Rate+, etc.)
- **Advanced passing** — air yards, pocket time, play action, pressure rates
- **Rushing & receiving** — dual-threat metrics
- **Defense & fumbles** — sacks taken, fumbles
- **Snap counts** — offensive/defensive/ST snap percentages
- **Elo ratings** — QB Elo, QBR, CPOE from external rankings

In [ ]:
print(f"QB Dataset: {qb.shape[0]} rows × {qb.shape[1]} columns")
print(f"Seasons: {int(qb['Season'].min())} – {int(qb['Season'].max())}")
print(f"Unique QBs: {qb['Player'].nunique()}")
print(f"\nFirst 5 rows (core columns):")
core_cols = ['Player', 'Season', 'Age', 'Team', 'G', 'GS', 'QBrec',
             'Cmp', 'Att', 'Cmp%', 'Yds', 'TD', 'Int', 'Rate', 'QBR']
qb[core_cols].head(10)

In [ ]:
# QB column groups
print("QB Column Groups:")
print(f"  Identity:          Player, PlayerID, Season, Age, Team, Pos, G, GS")
print(f"  Core Passing:      QBrec, Cmp, Att, Cmp%, Yds, TD, TD%, Int, Int%, 1D, Succ%, Lng, Y/A, AY/A, Y/C, Y/G, Rate, QBR, Sk, ...")
print(f"  Adjusted (adj_):   {len([c for c in qb.columns if c.startswith('adj_')])} columns")
print(f"  Advanced (adv_pass_): {len([c for c in qb.columns if c.startswith('adv_pass_')])} columns")
print(f"  Rush/Rec (rr_):    {len([c for c in qb.columns if c.startswith('rr_')])} columns")
print(f"  Adv Rush/Rec:      {len([c for c in qb.columns if c.startswith('adv_rr_')])} columns")
print(f"  Defense (def_):    {len([c for c in qb.columns if c.startswith('def_')])} columns")
print(f"  Snap Counts (snp_): {len([c for c in qb.columns if c.startswith('snp_')])} columns")
print(f"  Elo (elo_):        {len([c for c in qb.columns if c.startswith('elo_')])} columns")
print(f"  TOTAL:             {len(qb.columns)} columns")

In [ ]:
# Key QB columns explained
qb_column_explanations = {
    'QBrec': 'Win-Loss-Tie record as starter (e.g. "12-5-0")',
    'Cmp': 'Completions',
    'Att': 'Pass attempts',
    'Cmp%': 'Completion percentage',
    'Yds': 'Passing yards',
    'TD': 'Passing touchdowns',
    'TD%': 'TD percentage per attempt',
    'Int': 'Interceptions thrown',
    'Int%': 'Interception percentage per attempt',
    '1D': 'First downs from passing',
    'Succ%': 'Successful play percentage',
    'Y/A': 'Yards per attempt',
    'AY/A': 'Adjusted yards per attempt (TDs/INTs weighted)',
    'Y/C': 'Yards per completion',
    'Y/G': 'Yards per game',
    'Rate': 'Passer rating (0–158.3 scale)',
    'QBR': 'Total QBR (ESPN metric, 0–100)',
    'Sk': 'Times sacked',
    'NY/A': 'Net yards per attempt (sacks included)',
    'ANY/A': 'Adjusted net yards per attempt',
    '4QC': 'Fourth quarter comebacks',
    'GWD': 'Game-winning drives',
    'AV': 'Approximate Value (PFR composite)',
    'adj_ANY/A+': 'Era-adjusted ANY/A (100 = league average)',
    'adj_Rate+': 'Era-adjusted passer rating (100 = league average)',
    'adv_pass_pass_air_yds': 'Total air yards on pass attempts',
    'adv_pass_pocket_time': 'Average pocket time (seconds)',
    'adv_pass_pass_on_target_pct': 'On-target throw percentage',
    'adv_pass_pass_poor_throw_pct': 'Bad throw percentage',
    'adv_pass_pass_pressured_pct': 'Percentage of dropbacks under pressure',
}

print(f"{'Column':<32} Description")
print("=" * 85)
for col, desc in qb_column_explanations.items():
    print(f"{col:<32} {desc}")

In [ ]:
# QB descriptive statistics
qb_stats_cols = ['Cmp', 'Att', 'Cmp%', 'Yds', 'TD', 'Int', 'Rate', 'Y/A', 'QBR', 'Sk']
print("QB Descriptive Statistics (core passing):")
qb[qb_stats_cols].describe().round(2)

In [ ]:
# QB missing values analysis
print("QB Missing Values (top 20 columns with nulls):")
missing = qb.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False).head(20)
print(f"\n{'Column':<35} {'Missing':>8} {'% Missing':>10}")
print("-" * 55)
for col, count in missing.items():
    pct = count / len(qb) * 100
    print(f"{col:<35} {count:>8} {pct:>9.1f}%")
print(f"\nColumns with zero nulls: {(qb.isnull().sum() == 0).sum()} / {len(qb.columns)}")

In [ ]:
# QB data types
print("QB Data Types:")
print(qb.dtypes.value_counts().to_string())
print(f"\nSample of unique 'Pos' values: {qb['Pos'].dropna().unique()[:10]}")
print(f"Sample of 'QBrec' format: {qb['QBrec'].dropna().iloc[:5].tolist()}")

---
## 6. Running Back (RB) Dataset — Preview

**Source:** Pro Football Reference (scraped)  
**Players:** 77 RBs | **Seasons:** 1988–2025 | **Columns:** 113

Key column groups:
- **Rushing** — attempts, yards, TDs, success rate, YPC
- **Receiving** — targets, receptions, yards, TDs (dual-threat backs)
- **Advanced** (`adv_`) — yards before contact, yards after contact, broken tackles
- **Defense** (`def_`) — fumbles, tackles
- **Snap Counts** (`snp_`) — offensive/ST snap share

In [ ]:
print(f"RB Dataset: {rb.shape[0]} rows × {rb.shape[1]} columns")
print(f"Seasons: {int(rb['Season'].min())} – {int(rb['Season'].max())}")
print(f"Unique RBs: {rb['Player'].nunique()}")
print(f"\nFirst 10 rows (core columns):")
rb_core = ['Player', 'Season', 'Age', 'Team', 'G', 'GS',
           'Rush_Att', 'Rush_Yds', 'Rush_TD', 'Rush_Y/A',
           'Tgt', 'Rec', 'Rec_Yds', 'Rec_TD']
rb[rb_core].head(10)

In [ ]:
# RB key columns explained
rb_column_explanations = {
    'Rush_Att': 'Rushing attempts',
    'Rush_Yds': 'Rushing yards',
    'Rush_TD': 'Rushing touchdowns',
    'Rush_1D': 'First downs from rushing',
    'Rush_Succ%': 'Percentage of rushes gaining expected yards',
    'Rush_Y/A': 'Yards per rush attempt',
    'Rush_Y/G': 'Rushing yards per game',
    'Tgt': 'Receiving targets',
    'Rec': 'Receptions',
    'Rec_Yds': 'Receiving yards',
    'Rec_TD': 'Receiving touchdowns',
    'Catch%': 'Catch percentage (Rec / Tgt)',
    'Touches': 'Total touches (Rush_Att + Rec)',
    'Y/Touch': 'Yards per touch',
    'Scrimmage_Yds': 'Total scrimmage yards (Rush + Rec)',
    'adv_Rush_YBC': 'Yards before contact',
    'adv_Rush_YBC/A': 'Yards before contact per attempt',
    'adv_Rush_YAC': 'Yards after contact',
    'adv_Rush_BrkTkl': 'Broken tackles on rushes',
    'adv_Rec_aDOT': 'Average depth of target on receptions',
}

print(f"{'Column':<25} Description")
print("=" * 75)
for col, desc in rb_column_explanations.items():
    print(f"{col:<25} {desc}")

In [ ]:
# RB descriptive stats
rb_num = ['Rush_Att', 'Rush_Yds', 'Rush_TD', 'Rush_Y/A', 'Tgt', 'Rec', 'Rec_Yds', 'Rec_TD', 'Scrimmage_Yds']
rb_num = [c for c in rb_num if c in rb.columns]
print("RB Descriptive Statistics:")
rb[rb_num].describe().round(2)

---
## 7. Tight End (TE) Dataset — Preview

**Source:** Pro Football Reference (scraped)  
**Players:** 27 TEs | **Seasons:** 2013–2025 | **Columns:** 60

Tight ends are a hybrid position — primarily receivers but sometimes blockers/rushers. Key columns:
- **Receiving** — targets, receptions, yards, TDs, catch rate
- **Rushing** — limited carries for some TEs
- **Advanced** (`adv_`) — YAC, broken tackles, ADOT, passer rating when targeted

In [ ]:
print(f"TE Dataset: {te.shape[0]} rows × {te.shape[1]} columns")
print(f"Seasons: {int(te['Season'].min())} – {int(te['Season'].max())}")
print(f"Unique TEs: {te['Player'].nunique()}")
print(f"\nFirst 10 rows (core columns):")
te_core = ['Player', 'Season', 'Age', 'Team', 'G', 'GS',
           'Tgt', 'Rec', 'Rec_Yds', 'Rec_TD', 'Rec_Y/R', 'catch_pct']
te_core = [c for c in te_core if c in te.columns]
te[te_core].head(10)

In [ ]:
# TE key columns explained
te_column_explanations = {
    'Tgt': 'Receiving targets',
    'Rec': 'Receptions',
    'Rec_Yds': 'Receiving yards',
    'Rec_Y/R': 'Yards per reception',
    'Rec_TD': 'Receiving touchdowns',
    'Rec_1D': 'First downs from receiving',
    'catch_pct': 'Catch percentage',
    'Rec_Y/G': 'Receiving yards per game',
    'adv_ADOT': 'Average depth of target',
    'adv_Rec_YAC': 'Yards after catch',
    'adv_Rec_YAC/R': 'YAC per reception',
    'adv_Rec_YBC': 'Yards before catch (air yards per reception)',
    'adv_Rec_BrkTkl': 'Broken tackles after catch',
    'adv_rec_pass_rating': 'Passer rating when targeted',
    'adv_Drop': 'Drops',
    'adv_Drop%': 'Drop percentage',
}

print(f"{'Column':<25} Description")
print("=" * 70)
for col, desc in te_column_explanations.items():
    print(f"{col:<25} {desc}")

In [ ]:
te_num = ['Tgt', 'Rec', 'Rec_Yds', 'Rec_TD', 'Rec_Y/R', 'catch_pct']
te_num = [c for c in te_num if c in te.columns]
print("TE Descriptive Statistics:")
te[te_num].describe().round(2)

---
## 8. Wide Receiver (WR) Dataset — Preview

**Source:** HuggingFace — [SebastianAndreu/24679_NFL_WR_Dataset_2025](https://huggingface.co/datasets/SebastianAndreu/24679_NFL_WR_Dataset_2025)  
**Players:** 1,617 WRs | **Seasons:** 2015–2025 | **Columns:** 99

This dataset stands apart from the others — it's derived from **nflverse play-by-play data** rather than PFR page scrapes. It includes:
- Standard receiving stats (targets, receptions, yards, TDs)
- **Advanced analytics** — EPA, WPA, CPOE
- **Situational splits** — red zone, quarter-by-quarter, win probability buckets
- **Contextual data** — game spread, total, weather conditions
- **QB context** — QB attempts, air yards, CPOE

We have **two versions**:
- `wr_all_weeks.csv` — 46,115 rows (game-level, one row per player-game)
- `wr_all_seasons.csv` — 5,529 rows (season-level, aggregated per player-season)

In [ ]:
print(f"WR Season Dataset: {wr.shape[0]} rows × {wr.shape[1]} columns")
print(f"Seasons: {int(wr['season'].min())} – {int(wr['season'].max())}")
print(f"Unique WRs: {wr['receiver_player_name'].nunique()}")
print(f"\nWR Weekly Dataset: {wr_weeks.shape[0]} rows × {wr_weeks.shape[1]} columns")
print(f"\nFirst 10 rows (core columns):")
wr_core = ['receiver_player_name', 'season', 'games_played', 'posteam',
           'targets', 'receptions', 'receiving_yards', 'tds',
           'catch_rate', 'epa', 'air_yards', 'yac']
wr[wr_core].sort_values('receiving_yards', ascending=False).head(10)

In [ ]:
# WR key columns explained
wr_column_explanations = {
    'receiver_player_name': 'Player name',
    'receiver_player_id': 'Unique player identifier (nflverse)',
    'season': 'NFL season year',
    'games_played': 'Number of games played',
    'targets': 'Times targeted by quarterback',
    'receptions': 'Completed receptions',
    'receiving_yards': 'Total receiving yards',
    'tds': 'Receiving touchdowns',
    'air_yards': 'Total air yards on targets',
    'yac': 'Yards after catch',
    'epa': 'Expected Points Added (average per play)',
    'wpa': 'Win Probability Added (average per play)',
    'catch_rate': 'Receptions / Targets',
    'adot': 'Average depth of target',
    'target_share': 'Share of team targets',
    'air_yard_share': 'Share of team air yards',
    'red_zone_targets': 'Targets inside the 20-yard line',
    'explosive_plays': 'Plays gaining 20+ yards',
    'success_rate': 'Percentage of plays with positive EPA',
    'big_play_rate': 'Percentage of explosive plays',
    'pregame_spread': 'Pregame point spread (avg)',
    'temp_f': 'Temperature (F) average for games played',
    'wind_mph': 'Wind speed (mph) average',
}

print(f"{'Column':<28} Description")
print("=" * 80)
for col, desc in wr_column_explanations.items():
    print(f"{col:<28} {desc}")

In [ ]:
# WR descriptive stats
wr_num = ['targets', 'receptions', 'receiving_yards', 'tds', 'catch_rate', 'epa', 'adot', 'target_share']
wr_num = [c for c in wr_num if c in wr.columns]
print("WR Season-Level Descriptive Statistics:")
wr[wr_num].describe().round(2)

---
## 9. QB Elo Ratings Dataset — Preview

In addition to PFR stats, we merged **QB Elo ratings** — a composite ranking system that evaluates quarterback performance using a chess-like rating method.

Two files:
- `qb_rankings_career.csv` — 979 QB-seasons from 2009–2025 (239 unique QBs)
- `qb_rankings_2025.csv` — Current 2025 season ratings for 63 QBs

In [ ]:
print(f"ELO Career: {elo_career.shape[0]} rows × {elo_career.shape[1]} columns")
print(f"Seasons: {int(elo_career['Season'].min())} – {int(elo_career['Season'].max())}")
print(f"Unique QBs: {elo_career['QB'].nunique()}")
print(f"\nELO 2025: {elo_2025.shape[0]} rows × {elo_2025.shape[1]} columns")
print(f"\nTop 10 QBs by career Elo:")
elo_top = elo_career.groupby('QB')['QB Elo'].max().sort_values(ascending=False).head(10)
print(elo_top.to_string())

In [ ]:
print("ELO Columns:")
for i, col in enumerate(elo_career.columns, 1):
    print(f"  {i:2d}. {col}")

---
## 10. Complete Data Pipeline Summary

```
┌──────────────────────────────────────────────────────────────────────────┐
│                        DATA COLLECTION                                  │
│                                                                         │
│   Pro Football Reference              HuggingFace / nflverse            │
│   ┌────────────────────┐              ┌────────────────────┐            │
│   │ scrapers/           │              │ Pre-built dataset  │            │
│   │  nfl_qb_scraper.py │              │ WR weekly data     │            │
│   │  nfl_rb_scraper.py │              │ 2015–2025          │            │
│   │  nfl_te_scraper.py │              └────────┬───────────┘            │
│   └────────┬───────────┘                       │                        │
│            │                                   │                        │
│            v                                   v                        │
│   ┌────────────────────┐              ┌────────────────────┐            │
│   │ data/raw/           │              │ data/raw/wr/       │            │
│   │  qb/{player}/*.csv │              │  {year}/data/      │            │
│   │  rb/{player}/*.csv │              │  wr_{year}.csv     │            │
│   │  te/{player}/*.csv │              └────────┬───────────┘            │
│   └────────┬───────────┘                       │                        │
│            │                                   │                        │
│            v                                   v                        │
│   ┌──────────────────────────────────────────────────────────┐          │
│   │                scripts/ (combining)                       │          │
│   │  combine_qb_master.py   combine_rb_master.py             │          │
│   │  combine_te_master.py   combine_wr_master.py             │          │
│   └────────────────────┬─────────────────────────────────────┘          │
│                        │                                                │
│                        v                                                │
│   ┌──────────────────────────────────────────────────────────┐          │
│   │              data/fully combined/                         │          │
│   │  qb_master.csv      (845 × 142)                         │          │
│   │  rb_master.csv      (622 × 113)                         │          │
│   │  te_master.csv      (194 × 60)                          │          │
│   │  wr_all_seasons.csv (5529 × 99)                         │          │
│   │  wr_all_weeks.csv   (46115 × 99)                        │          │
│   └──────────────────────────────────────────────────────────┘          │
│                                                                         │
│   + data/nfl elo data/                                                  │
│     qb_rankings_career.csv (979 × 46)                                  │
│     qb_rankings_2025.csv   (63 × 46)                                   │
└──────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Final summary table
print("=" * 70)
print("DATASET SUMMARY")
print("=" * 70)
summary_data = {
    'Position': ['QB', 'RB', 'TE', 'WR (season)', 'WR (weekly)', 'QB Elo'],
    'File': ['qb_master.csv', 'rb_master.csv', 'te_master.csv',
             'wr_all_seasons.csv', 'wr_all_weeks.csv', 'qb_rankings_career.csv'],
    'Rows': [len(qb), len(rb), len(te), len(wr), len(wr_weeks), len(elo_career)],
    'Columns': [len(qb.columns), len(rb.columns), len(te.columns),
                len(wr.columns), len(wr_weeks.columns), len(elo_career.columns)],
    'Players': [qb['Player'].nunique(), rb['Player'].nunique(),
                te['Player'].nunique(), wr['receiver_player_name'].nunique(),
                wr_weeks['receiver_player_name'].nunique(), elo_career['QB'].nunique()],
    'Season Range': [
        f"{int(qb['Season'].min())}–{int(qb['Season'].max())}",
        f"{int(rb['Season'].min())}–{int(rb['Season'].max())}",
        f"{int(te['Season'].min())}–{int(te['Season'].max())}",
        f"{int(wr['season'].min())}–{int(wr['season'].max())}",
        f"{int(wr_weeks['season'].min())}–{int(wr_weeks['season'].max())}",
        f"{int(elo_career['Season'].min())}–{int(elo_career['Season'].max())}",
    ],
    'Source': ['PFR (scraped)', 'PFR (scraped)', 'PFR (scraped)',
               'HuggingFace', 'HuggingFace', 'Elo rankings']
}
summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print("\nAll data loaded and ready for analysis in Notebook 2 (EDA) and Notebook 3 (Models).")